<a href="https://colab.research.google.com/github/Satish-Kumar-1/RAG-/blob/main/youtube_chatbot_RAG_Lanchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [2]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_6218/3829014579.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# Step 1a - Indexing (Document Ingestion)

In [7]:
video_id = "Gfr50f6ZBvo"
try:
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(video_id=video_id)
    transcript = " ".join(snippet.text for snippet in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

RequestBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to aut

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks), chunks[100].page_content

(168,
 "and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask what's involved in that task 

# Step2 - Embedding and Stroing in Vector DB


In [ ]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id, vector_store.get_by_ids(['c1638350-3c2f-4bf9-920c-989aa9139a6c'])

({0: 'c1638350-3c2f-4bf9-920c-989aa9139a6c',
  1: '0b2658a1-7693-406d-9e97-89b09fab055c',
  2: '23a1c24a-9719-4fe6-bbbf-4f5b141dc922',
  3: '7148a08a-2f23-4a9e-b2f3-9b6a1a5e68e2',
  4: 'cad9cb27-b542-4250-922e-c4fd883045dc',
  5: '6864e2ac-6c10-48af-87d3-5afa8bbc74f1',
  6: '0e00d94b-7d92-4b6a-869e-f25abcf03b5b',
  7: '880a20d4-5119-4125-b81a-f8a16195b8c5',
  8: '08385279-5ed8-4ec4-a218-10c6a42be469',
  9: '21b8ed6c-e629-441d-b312-9e7405fd068d',
  10: 'f32f3073-b053-45b3-b983-afb142b80e54',
  11: '87b0e24e-1bcf-40b4-a67d-26017dbd3e87',
  12: '940a8b99-dd29-48b5-bf9d-45eba96d529b',
  13: '0820b605-b211-402f-921c-9920730c3415',
  14: '20a8412d-c6aa-402f-9483-20fdf0350d99',
  15: 'cd5d717a-561a-4f53-b39b-7e39a5bece17',
  16: '1cf4e964-9e6c-4ef9-9c14-1c95922ba1e5',
  17: '57300fa1-5428-4de7-929e-eb4052bd59c5',
  18: 'a753c382-03d7-488a-badb-fb10b01f13ae',
  19: 'd1835953-9fa2-4e59-a492-c1df6aa3bd68',
  20: 'eaaf3e6b-d34e-4ae8-a9c6-9a90591406ba',
  21: 'e82b5864-87e0-4d1d-9d86-cd51d8c366db'

# Retrieval

In [ ]:
retreiver = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k" : 4})

In [ ]:
retreiver

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b0924127950>, search_kwargs={'k': 4})

In [ ]:
retreiver.invoke("What is deepmind")

[Document(id='c1638350-3c2f-4bf9-920c-989aa9139a6c', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

#Augmentation

In [ ]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0.2)

In [ ]:
prompt = PromptTemplate(
    template = """
        you are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.

        {context}
        Question : {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "is the topic of nuclear fusion discusses in the video? if yes then what was discusses"
retrieved_docs = retreiver.invoke(question)

In [ ]:
retrieved_docs

[Document(id='b2133ea8-8f42-4b27-8629-46e906cf81f1', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({"context":context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text="\n        you are a helpful assistant. \n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n\n        in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottleneck

#Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed in the video. The discussion includes collaboration with EPFL in Switzerland, where they have a test reactor that was used for experiments. The focus is on identifying bottleneck problems in fusion and applying AI methods to address these challenges. One specific achievement mentioned is the ability to hold plasma in specific shapes for extended periods, which is crucial for energy production. The conversation also touches on the potential of AI to help solve various challenges in fusion, including physics, material science, and engineering issues related to building fusion reactors and containing plasma. Additionally, there is mention of ongoing discussions with fusion startups to tackle the next problems in the fusion area.


## Building a Chain

In [ ]:
from langchain_core.runnables import  RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context' : retreiver | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("Who is Demis")

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("Can you summarize the video")

'The video features a conversation between Lex and Demas, discussing the limitations of the standard model of physics and the need for more fundamental explanations. They explore topics such as consciousness, life, and gravity, emphasizing the importance of finding simpler and deeper explanations for these mysteries. Demas shares insights about his work in fusion energy, specifically how he has developed a controller to hold plasma in specific shapes for extended periods, which is a significant advancement in fusion research. They also touch on the modeling of quantum mechanical behavior of electrons and the potential role of AI in this area. The conversation concludes with a quote about the broader implications of computer science.'